# NB03: Explainability + Sanity Checks

Hard stop if sanity checks fail.

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not in Colab — skip Drive mount.')

## RUN MODE — edit the cell below (only switch in this notebook)

| `XFAD_DRY_RUN` | Meaning |
|---|---|
| `'1'` | **Smoke test** — tiny samples, 1 epoch, fast sanity check (~minutes) |
| `'0'` | **Full paper run** — all data, full training, both student arms (~hours) |

On a **full run**, NB02–NB05 automatically process **baseline** and **saliency** students (everything NB06 needs).

Then: **Runtime → Run all** (restart session after changing the switch).


In [ ]:
import os

# ╔══════════════════════════════════════════════════════════════════════╗
# ║  ONLY SWITCH IN THIS NOTEBOOK                                        ║
# ║  '0' = FULL PAPER RUN (all data, both student arms)                  ║
# ║  '1' = SMOKE TEST (tiny data, baseline student only)                 ║
# ╚══════════════════════════════════════════════════════════════════════╝
os.environ['XFAD_DRY_RUN'] = '0'


In [ ]:
import os
import sys
from pathlib import Path


def bootstrap_xfad_root() -> Path:
    """Locate repo root and put it on sys.path (works in Colab + local)."""
    candidates = []
    env = os.environ.get('XFAD_ROOT')
    if env:
        candidates.append(Path(env))
    candidates.extend([
        Path('/content/drive/MyDrive/XFAD'),
        Path.cwd(),
        Path.cwd().parent,
    ])
    candidates.extend(Path.cwd().resolve().parents)
    seen = set()
    for p in candidates:
        try:
            p = p.resolve()
        except OSError:
            continue
        if p in seen:
            continue
        seen.add(p)
        if (p / 'src' / 'config.py').exists():
            if str(p) not in sys.path:
                sys.path.insert(0, str(p))
            os.environ.setdefault('XFAD_ROOT', str(p))
            return p
    raise FileNotFoundError(
        'XFAD repo root not found (expected src/config.py). '
        'In Colab: mount Drive first, then either keep the project at '
        '/content/drive/MyDrive/XFAD or set '
        "os.environ['XFAD_ROOT'] = '/content/drive/MyDrive/<your>/XFAD'."
    )


REPO_ROOT = bootstrap_xfad_root()
print('REPO_ROOT:', REPO_ROOT)

from src.utils.io import set_seed
set_seed(42)

from src.utils.io import resolve_dry_run, print_run_banner, student_variants_for_run
DRY_RUN = resolve_dry_run()
import torch
from src.config import COMMON_FS, MITBIH_DS1_VAL, STUDENT_TRUNK_CHANNELS
from src.data.mitbih import MITBIHDataset
from src.explain.gradcam import grad_cam_1d
from src.explain.faithfulness import deletion_auc, pointing_game_ratios
from src.models.student import MultiTaskStudent

device = torch.device('cuda' if torch.cuda.is_available() and not DRY_RUN else 'cpu')

STUDENT_VARIANTS_TO_RUN = student_variants_for_run(DRY_RUN)
print_run_banner(notebook='NB03_Explainability', dry_run=DRY_RUN, variants=STUDENT_VARIANTS_TO_RUN)

In [ ]:
import json
import numpy as np
from src.config import RESULTS_DIR
from src.explain.sanity_checks import parameter_randomization_report, evaluate_sanity_checks
from src.utils.io import ensure_dir, student_checkpoint_path, variant_result_path

ensure_dir(RESULTS_DIR)

ds = MITBIHDataset(MITBIH_DS1_VAL, dry_run=DRY_RUN, use_cache=True)
print(f'Validation beats loaded: {len(ds)} (dry_run={DRY_RUN})')

beat_len = ds[0][0].shape[-1]
beat_center = beat_len // 2
qrs_half = int(0.04 * COMMON_FS)  # ~40 ms either side of the annotated R peak
qrs_onset, qrs_offset = beat_center - qrs_half, beat_center + qrs_half

n_examples = 2 if DRY_RUN else 50
n_faith = 2 if DRY_RUN else 50
steps = 10 if DRY_RUN else 50
sanity_indices = torch.randperm(len(ds), generator=torch.Generator().manual_seed(42))[:n_examples].tolist()
faith_indices = torch.randperm(len(ds), generator=torch.Generator().manual_seed(7))[:n_faith].tolist()


def mean_sd(values):
    arr = np.asarray(values, dtype=float)
    return {'mean': float(arr.mean()), 'sd': float(arr.std(ddof=1)) if len(arr) > 1 else 0.0}


for VARIANT in STUDENT_VARIANTS_TO_RUN:
    print('\n' + '=' * 60 + f'\nStudent variant: {VARIANT}\n' + '=' * 60)
    STUDENT_PATH = student_checkpoint_path(VARIANT)
    if not os.path.exists(STUDENT_PATH):
        raise FileNotFoundError(f'{STUDENT_PATH} missing — run NB02 first (full run trains both arms).')
    print(f'Checkpoint: {STUDENT_PATH}')

    student = MultiTaskStudent(c_internal=STUDENT_TRUNK_CHANNELS).to(device)
    student.load_state_dict(torch.load(STUDENT_PATH, map_location=device))
    student.eval()

    x, y = ds[0]
    x = x.unsqueeze(0).to(device)
    pred = int(student(x, 'A').argmax(dim=1).item())
    cam = grad_cam_1d(student, x, pred, MultiTaskStudent.LAST_CONV, task='A')
    assert cam.shape[-1] == x.shape[-1]
    print('Grad-CAM shape OK')

    cosine_sims, rank_sims = [], []
    for k, idx in enumerate(sanity_indices):
        xi, yi = ds[idx]
        xi = xi.unsqueeze(0).to(device)
        report = parameter_randomization_report(
            student, xi, int(yi), MultiTaskStudent.LAST_CONV, task='A',
            grad_cam_fn=grad_cam_1d, seed=1000 + k,
        )
        cosine_sims.append(report['cosine'])
        rank_sims.append(report['spearman'])

    status = evaluate_sanity_checks(rank_sims, metric='spearman')
    sanity = {
        'n_examples': n_examples,
        'spearman': {
            'mean': float(np.mean(rank_sims)),
            'median': float(np.median(rank_sims)),
            'max': float(np.max(rank_sims)),
            'status': status,
        },
        'cosine': {
            'mean': float(np.mean(cosine_sims)),
            'median': float(np.median(cosine_sims)),
            'max': float(np.max(cosine_sims)),
            'status': evaluate_sanity_checks(cosine_sims, metric='cosine'),
        },
    }
    print(json.dumps(sanity, indent=2))

    if status == 'fail':
        if not DRY_RUN:
            raise RuntimeError('Sanity check FAILED: randomised weights reproduce the Grad-CAM - fix before NB04')
        print('WARNING: smoke-test student may fail sanity; enforced only when DRY_RUN=False.')
    elif status == 'inconclusive':
        print('WARNING: inconclusive - report the similarity distribution rather than a pass/fail claim')

    cam_aucs, random_aucs, ratio_rows = [], [], []
    for idx in faith_indices:
        xi, _ = ds[idx]
        xi = xi.unsqueeze(0).to(device)
        pred = int(student(xi, 'A').argmax(dim=1).item())
        cam = grad_cam_1d(student, xi, pred, MultiTaskStudent.LAST_CONV, task='A')[0]
        cam_aucs.append(deletion_auc(student, xi, pred, cam, steps=steps, task='A'))
        random_aucs.append(deletion_auc(student, xi, pred, torch.rand_like(cam), steps=steps, task='A'))
        ratio_rows.append(pointing_game_ratios(cam, qrs_onset, qrs_offset, COMMON_FS))

    faithfulness = {
        'n_examples': n_faith,
        'deletion_steps': steps,
        'qrs_window_samples': [int(qrs_onset), int(qrs_offset)],
        'deletion_auc_gradcam': mean_sd(cam_aucs),
        'deletion_auc_random': mean_sd(random_aucs),
        'pointing_ratios': {k: mean_sd([r[k] for r in ratio_rows]) for k in ('p_ratio', 'qrs_ratio', 't_ratio')},
    }
    faithfulness['deletion_auc_advantage_over_random'] = (
        faithfulness['deletion_auc_random']['mean'] - faithfulness['deletion_auc_gradcam']['mean']
    )
    print(json.dumps(faithfulness, indent=2))

    explainability_path = variant_result_path('explainability_eval', VARIANT)
    with open(explainability_path, 'w', encoding='utf-8') as f:
        json.dump({
            'dry_run': DRY_RUN,
            'student_variant': VARIANT,
            'sanity_checks': sanity,
            'faithfulness': faithfulness,
        }, f, indent=2)
    print('Saved', explainability_path)
